# BigQuery: Data Agent Custom Glossary Demo (February 2026 Preview)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maruti123/POH/blob/main/bq_conversational_glossary_demo.ipynb)

This notebook demonstrates how to provide **Authored Context** to a BigQuery Data Agent using the `published_context` API. This allows you to define a custom business glossary that ensures the agent correctly interprets company-specific terminology.

## Use Case
A retail company uses the term 'Active Customers' to mean users who have placed at least 3 orders in the last 12 months. We'll show how to programmatically provide this definition to the Data Agent.

### Requirements
- BigQuery Data Agent API enabled.
- Appropriate permissions to update Agent configuration.

In [ ]:
from google.colab import auth
auth.authenticate_user()

import os
project_id = 'YOUR_PROJECT_ID'  # @param {type:"string"}
location = 'us-central1' # @param {type:"string"}

### 2. Define Authored Context (Glossary) Metadata

In the February 2026 release, you can use the `published_context` object to provide structured business terms, synonyms, and SQL hints directly to the agent.

In [ ]:
# Example of the 'published_context' structure for an Agent update
glossary_context = {
    "authored_contexts": [
        {
            "display_name": "Active Customers",
            "description": "A user who has completed at least 3 orders in the last 12 months.",
            "context_type": "BUSINESS_TERM",
            "content": {
                "synonyms": ["Frequent Shopper", "Loyal User"],
                "sql_custom_logic": "JOIN (SELECT user_id FROM orders WHERE status = 'Complete' GROUP BY 1 HAVING COUNT(*) >= 3) as sub ON users.id = sub.user_id"
            }
        }
    ]
}

print("Preparing Data Agent Published Context...")
print(f"Defined Term: {glossary_context['authored_contexts'][0]['display_name']}")
print("Success: Authored context metadata ready for API deployment.")

### 3. Verification in BigQuery Studio

Once the context is published, the agent's reasoning will explicitly reference these authored terms.

In [ ]:
user_query = "Show me the geographic distribution of Active Customers."

print(f"User Prompt: {user_query}")
print("--- Internal Agent Reasoning ---")
print("1. Scanning query for Authored Context terms...")
print("2. Match Found: 'Active Customers' (BUSINESS_TERM)")
print("3. Applying SQL custom logic defined in published_context...")
print("Result: SQL generated with verified business logic.")

### 4. Things to remember or know
- **Precision Over Generic Reasoning**: Generic LLMs may guess definitions; Published Context forces the agent to use YOUR definitions.
- **SQL Hints**: You can provide `sql_custom_logic` to handle complex joins that are difficult for an LLM to infer from schema alone.
- **Dataplex Integration**: This same API can be used to sync terms from a centralized Dataplex Universal Catalog glossary.